Sub processes. Additional note for us: After running the cell -> we need to restart the runtime and run from the next cell

In [ ]:
import subprocess, sys

r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install',
     'torch==2.1.0', 'torchvision==0.16.0', 'torchaudio==2.1.0',
     '--index-url', 'https://download.pytorch.org/whl/cu118', '-q'],
    capture_output=True, text=True
)
print(r.stdout[-300:] if r.stdout else "install complete.")

install complete.

done. now: Runtime -> Restart session -> then run Cell 1.


Dependencies & constants

In [ ]:
import os, glob, json, random, io, time, threading
import numpy as np
import cv2
import dlib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, Image as IPImage

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from jiwer import wer as jiwer_wer

SEED = 42
NUM_SPEAKERS = 4 # 4 for testing, 33 for full run (s21 excluded, audio-only)

# frame dimensions (paper Table 1)
T = 75
H = 46
W = 140

# mouth crop padding (paper section III-B)
H_PAD = 0.30
V_PAD = 0.50

# dlib landmark indices for the mouth
OUTER_LIP = list(range(48, 60))   # 12 points, closed loop
INNER_LIP = list(range(60, 68))   # 8 points, closed loop

# vocab: 40 characters + 1 CTC blank = 41 classes
CHARS = list("abcdefghijklmnopqrstuvwxyz0123456789 ?!'")
CHAR2IDX = {c: i + 1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}
VOCAB_SIZE = len(CHARS)
NUM_CLASSES = VOCAB_SIZE + 1
BLANK_INDEX = 0

# training
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
EPOCHS = 100

# paths (all on Drive so nothing is lost on crash)
PROJECT = '/content/drive/MyDrive/lipsyncnet_baseline'
GRID_RAW = f'{PROJECT}/grid_raw'
GRID_ROOT = f'{GRID_RAW}/data'
GRID_OUT = f'{PROJECT}/grid_processed'
CKPT_DIR = f'{PROJECT}/checkpoints'
PLOT_DIR = f'{PROJECT}/plots'
SPEAKER_LIST_PATH = f'{PROJECT}/selected_speakers.npy'
PREDICTOR_PATH = 'shape_predictor_68_face_landmarks.dat'

KAGGLE_DATASET = "ramakrishnasonakam/data-uploading-test"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

Mounting, dependencies, dlib, kaggle connection, speakers



In [ ]:
def mount_drive():
    from google.colab import drive
    drive.mount('/content/drive')
    for d in [GRID_OUT, CKPT_DIR, PLOT_DIR]:
        os.makedirs(d, exist_ok=True)
    print(f"drive mounted. project: {PROJECT}")


def install_deps():
    """install dlib, opencv, jiwer. only runs once per session."""
    get_ipython().system('apt-get install -y cmake libboost-all-dev -q')
    get_ipython().system('pip install dlib opencv-python-headless jiwer -q')
    print("dependencies installed.")


def download_dlib_predictor():
    if os.path.exists(PREDICTOR_PATH):
        print("dlib predictor already present.")
        return
    get_ipython().system('wget -q http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2')
    get_ipython().system('bzip2 -d shape_predictor_68_face_landmarks.dat.bz2')
    print("dlib predictor downloaded.")


def setup_kaggle():
    from google.colab import userdata
    creds = {"username": userdata.get('KAGGLE_USERNAME'), "key": userdata.get('KAGGLE_KEY')}
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(creds, f)
    get_ipython().system('chmod 600 /root/.kaggle/kaggle.json')
    print("kaggle credentials set.")


def download_grid():
    """download GRID to Drive. skips if already there."""
    if os.path.exists(GRID_ROOT) and len(os.listdir(GRID_ROOT)) > 0:
        print(f"grid data already on drive, skipping download.")
        return
    os.makedirs(GRID_RAW, exist_ok=True)
    get_ipython().system(f'kaggle datasets download -d {KAGGLE_DATASET} -p {GRID_RAW} --unzip')
    print("grid download complete.")


def start_heartbeat():
    """dot every 30s to prevent colab timeout. runs in background."""
    def tick():
        while True:
            print(".", end="", flush=True)
            time.sleep(30)
    threading.Thread(target=tick, daemon=True).start()
    print("heartbeat started.")


def list_available_speakers():
    """print what's in the download so we can see what we're working with."""
    speakers = sorted([d for d in os.listdir(GRID_ROOT) if os.path.isdir(os.path.join(GRID_ROOT, d))])
    print(f"\nspeakers found: {len(speakers)}")
    for spk in speakers:
        v = len(glob.glob(f'{GRID_ROOT}/{spk}/*.mpg'))
        a = len(glob.glob(f'{GRID_ROOT}/{spk}/align/*.align'))
        print(f"  {spk} — {v} videos | {a} aligns")
    return speakers


def select_speakers(available):
    """
    pick speakers and lock the selection to Drive for consistency.
    s21 is always excluded (audio-only in GRID).
    if a saved selection exists AND matches NUM_SPEAKERS, reuse it.
    otherwise pick fresh and save.
    """
    available = [s for s in available if s != 's21']

    if os.path.exists(SPEAKER_LIST_PATH):
        saved = list(np.load(SPEAKER_LIST_PATH, allow_pickle=True))
        if len(saved) == NUM_SPEAKERS:
            print(f"loaded saved speakers ({NUM_SPEAKERS}): {saved}")
            return saved
        else:
            print(f"saved selection has {len(saved)} speakers but we need {NUM_SPEAKERS}. re-selecting.")

    if len(available) < NUM_SPEAKERS:
        raise ValueError(f"need {NUM_SPEAKERS} speakers but only {len(available)} available.")

    random.seed(SEED)
    selected = sorted(random.sample(available, NUM_SPEAKERS))
    np.save(SPEAKER_LIST_PATH, selected)
    print(f"selected and saved ({NUM_SPEAKERS}): {selected}")
    return selected


def split_train_val(speakers):
    """last speaker = validation, rest = training."""
    val = speakers[-1]
    train = [s for s in speakers if s != val]
    print(f"train ({len(train)}): {train}")
    print(f"val   (1): {val}")
    return train, val


def load_dlib_models():
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor(PREDICTOR_PATH)
    print("dlib models loaded.")
    return detector, predictor

Preprocessing functions

In [ ]:
def get_mouth_roi(gray, landmarks):
    """crop mouth region (landmarks 48-67) with padding. returns (crop, bbox)."""
    pts = np.array([[landmarks.part(i).x, landmarks.part(i).y] for i in range(48, 68)])
    x0, y0 = pts.min(axis=0)
    x1, y1 = pts.max(axis=0)
    px = int((x1 - x0) * H_PAD)
    py = int((y1 - y0) * V_PAD)
    x0 = max(0, x0 - px)
    y0 = max(0, y0 - py)
    x1 = min(gray.shape[1], x1 + px)
    y1 = min(gray.shape[0], y1 + py)
    return gray[y0:y1, x0:x1], (x0, y0, x1, y1)


def read_label(align_path):
    """read .align file, drop silence/pause markers, return sentence string."""
    words = []
    with open(align_path) as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 3 and p[2] not in ('sil', 'sp'):
                words.append(p[2])
    return ' '.join(words)


def process_video(path, detector, predictor):
    """
    full pipeline for one clip: load -> detect face -> crop mouth -> normalize -> pad/trim.
    returns (75, 46, 140, 1) float32 array, or None on failure.
    """
    cap = cv2.VideoCapture(path)
    frames = []
    last_good_roi = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray, 0)

        if len(faces) == 0:
            if last_good_roi is not None:
                frames.append(last_good_roi.copy())
            else:
                frames.append(np.zeros((H, W), dtype=np.float32))
        else:
            roi, _ = get_mouth_roi(gray, predictor(gray, faces[0]))
            if roi.size == 0:
                frames.append(np.zeros((H, W), dtype=np.float32))
            else:
                roi = cv2.resize(roi, (W, H)).astype(np.float32)
                last_good_roi = roi
                frames.append(roi)

    cap.release()
    if len(frames) == 0:
        return None

    frames = np.array(frames, dtype=np.float32)
    if frames.std() > 0:
        frames = (frames - frames.mean()) / frames.std()

    if len(frames) < T:
        frames = np.concatenate([frames, np.zeros((T - len(frames), H, W), dtype=np.float32)])
    elif len(frames) > T:
        start = (len(frames) - T) // 2
        frames = frames[start:start + T]

    return frames[..., np.newaxis]


def preprocess_all(speakers, detector, predictor):
    """
    run preprocessing on all speakers. resume-safe — skips existing files.
    shows lip skeleton for first clip of each speaker.
    """
    total = 0
    skipped = 0
    failed = []
    vis_done = set()

    for spk in speakers:
        videos = sorted(glob.glob(f'{GRID_ROOT}/{spk}/*.mpg'))
        print(f"\n--- {spk} — {len(videos)} videos ---")
        spk_new = 0

        for vp in videos:
            name = os.path.splitext(os.path.basename(vp))[0]
            ap = f'{GRID_ROOT}/{spk}/align/{name}.align'
            op = f'{GRID_OUT}/{spk}_{name}.npz'

            if not os.path.exists(ap):
                continue
            if os.path.exists(op):
                skipped += 1
                continue

            try:
                tensor = process_video(vp, detector, predictor)
                label = read_label(ap)

                if tensor is None:
                    failed.append(f"{spk}/{name}: no frames")
                    continue

                np.savez_compressed(op, frames=tensor, label=label)
                total += 1
                spk_new += 1

                if spk not in vis_done:
                    try:
                        show_lip_skeleton(vp, f'{PLOT_DIR}/lip_skeleton_{spk}.png', detector, predictor)
                        print(f"  lip skeleton shown above ↑")
                    except Exception as ve:
                        print(f"  visualization skipped: {ve}")
                    vis_done.add(spk)

                if total % 100 == 0:
                    drive_n = len(glob.glob(f'{GRID_OUT}/*.npz'))
                    print(f"  progress: {total} new | {skipped} skipped | {drive_n} total on drive")

            except Exception as e:
                failed.append(f"{spk}/{name}: {e}")

        spk_total = len(glob.glob(f'{GRID_OUT}/{spk}_*.npz'))
        print(f"  {spk}: +{spk_new} this session | {spk_total} total on drive")

    drive_n = len(glob.glob(f'{GRID_OUT}/*.npz'))
    print(f"\n{'='*50}")
    print(f"PREPROCESSING DONE")
    print(f"  new this session: {total}")
    print(f"  skipped (resume): {skipped}")
    print(f"  failed:           {len(failed)}")
    print(f"  total on drive:   {drive_n}")

    if failed:
        print(f"\nfirst 20 failures:")
        for f in failed[:20]:
            print(f"  {f}")

    return total, failed

In [ ]:
def parse_lrs2_label(txt_path):

    with open(txt_path) as f:
        for line in f:
            if line.startswith('Text:'):
                return line.replace('Text:', '').strip().lower()
    return ''


def get_lrs2_clips(lrs2_root):

    import pathlib
    clips = []
    for speaker_dir in sorted(pathlib.Path(lrs2_root).iterdir()):
        if not speaker_dir.is_dir():
            continue
        speaker_id = speaker_dir.name
        for video_file in sorted(speaker_dir.glob('*.mp4')):
            clip_id  = video_file.stem
            txt_file = speaker_dir / f'{clip_id}.txt'
            if not txt_file.exists():
                print(f"  no .txt for {speaker_id}/{clip_id}, skipping")
                continue
            clips.append((speaker_id, clip_id, str(video_file), str(txt_file)))
    print(f"found {len(clips)} LRS2 clips")
    return clips


def preprocess_lrs2(detector, predictor):

    os.makedirs(LRS2_NPZ_DIR, exist_ok=True)
    all_clips = get_lrs2_clips(LRS2_RAW)

    if not all_clips:
        print(f"no clips found — check LRS2_RAW: {LRS2_RAW}")
        return

    total, skipped, failed = 0, 0, []

    for i, (speaker_id, clip_id, video_path, txt_path) in enumerate(all_clips):
        out_path = f'{LRS2_NPZ_DIR}/{speaker_id}_{clip_id}.npz'

        if os.path.exists(out_path):
            skipped += 1
            continue

        try:
            # reuse process_video — same pipeline as GRID
            tensor = process_video(video_path, detector, predictor)
            label  = parse_lrs2_label(txt_path)

            if tensor is None or not label:
                failed.append(f"{speaker_id}/{clip_id}: no frames or empty label")
                continue

            np.savez_compressed(out_path, frames=tensor, label=label)
            total += 1

        except Exception as e:
            failed.append(f"{speaker_id}/{clip_id}: {e}")

        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(all_clips)} — new:{total} skipped:{skipped} failed:{len(failed)}")

    lrs2_total = len(glob.glob(f'{LRS2_NPZ_DIR}/*.npz'))
    print(f"LRS2 done — new:{total} skipped:{skipped} failed:{len(failed)} total on drive:{lrs2_total}")
    if failed:
        print(f"failures: {failed[:10]}")

Visualisations

In [ ]:
def draw_lip_skeleton(frame_rgb, landmarks):
    """
    draw white-dot exoskeleton on a frame.
    outer lip (48-59) and inner lip (60-67) as two closed loops
    connected by white lines with white dots at each landmark.
    """
    img = frame_rgb.copy()

    # faint dots on all 68 landmarks for reference
    for i in range(68):
        cv2.circle(img, (landmarks.part(i).x, landmarks.part(i).y), 1, (255, 255, 255), -1)

    # outer lip: 48-59, closed loop
    outer_pts = [(landmarks.part(i).x, landmarks.part(i).y) for i in OUTER_LIP]
    for i in range(len(outer_pts)):
        cv2.line(img, outer_pts[i], outer_pts[(i + 1) % len(outer_pts)], (255, 255, 255), 2)
    for pt in outer_pts:
        cv2.circle(img, pt, 3, (255, 255, 255), -1)

    # inner lip: 60-67, closed loop
    inner_pts = [(landmarks.part(i).x, landmarks.part(i).y) for i in INNER_LIP]
    for i in range(len(inner_pts)):
        cv2.line(img, inner_pts[i], inner_pts[(i + 1) % len(inner_pts)], (255, 255, 255), 1)
    for pt in inner_pts:
        cv2.circle(img, pt, 2, (255, 255, 255), -1)

    return img


def show_lip_skeleton(video_path, save_path, detector, predictor):
    """
    visual proof that dlib is mapping lip shapes correctly.
    picks 6 evenly spaced frames, draws skeleton + crop box.
    saves to drive and displays inline.
    """
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return

    indices = [int(total * i / 6) for i in range(6)]
    fig, axes = plt.subplots(1, 6, figsize=(21, 4.2))
    fig.patch.set_facecolor('#0d0d1a')

    for col, fi in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        ax = axes[col]
        ax.set_facecolor('#0d0d1a')
        ax.axis('off')
        ax.set_title(f"frame {fi}", color='#888888', fontsize=8, pad=4)

        if not ret:
            continue

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = detector(gray, 0)

        if len(faces) == 0:
            ax.imshow(rgb)
            ax.text(0.5, 0.5, 'no face', color='red', ha='center', va='center', transform=ax.transAxes, fontsize=8)
            continue

        lm = predictor(gray, faces[0])
        ax.imshow(draw_lip_skeleton(rgb, lm))

        # green crop box
        _, bbox = get_mouth_roi(gray, lm)
        bx0, by0, bx1, by1 = bbox
        ax.add_patch(patches.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, linewidth=1.8, edgecolor='#00FF9F', facecolor='none', linestyle='--', zorder=6))

    spk = os.path.basename(os.path.dirname(video_path))
    clip = os.path.splitext(os.path.basename(video_path))[0]
    fig.suptitle(f"lip skeleton — {spk}/{clip}   outer(48-59) + inner(60-67)   □ crop region", color='white', fontsize=9, y=1.02)
    plt.tight_layout(pad=0.4)
    plt.savefig(save_path, dpi=140, bbox_inches='tight', facecolor='#0d0d1a', edgecolor='none')

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=110, bbox_inches='tight', facecolor='#0d0d1a', edgecolor='none')
    buf.seek(0)
    display(IPImage(data=buf.read()))
    plt.close()
    cap.release()

Verification functions

In [ ]:
def count_files_per_speaker():
    """count how many .npz files exist per speaker on drive."""
    all_files = sorted(glob.glob(f'{GRID_OUT}/*.npz'))
    counts = {}
    for f in all_files:
        spk = os.path.basename(f).split('_')[0]
        counts[spk] = counts.get(spk, 0) + 1

    print(f"total .npz files: {len(all_files)}")
    for spk in sorted(counts):
        print(f"  {spk}: {counts[spk]}")
    return all_files, counts


def sanity_check():
    """spot-check a sample file for correct shape, stats, and visual appearance."""
    files = sorted(glob.glob(f'{GRID_OUT}/*.npz'))
    if len(files) == 0:
        raise RuntimeError("no .npz files found. run preprocessing first.")

    sample = np.load(files[len(files) // 2], allow_pickle=True)
    print(f"\nshape: {sample['frames'].shape}   (expect: (75, 46, 140, 1))")
    print(f"label: '{sample['label']}'")
    print(f"mean:  {sample['frames'].mean():.3f}   (expect: ~0.0)")
    print(f"std:   {sample['frames'].std():.3f}    (expect: ~1.0)")

    # show 10 evenly spaced frames
    fig, axes = plt.subplots(1, 10, figsize=(22, 3))
    fig.patch.set_facecolor('#0d0d1a')
    for i, ax in enumerate(axes):
        ax.imshow(sample['frames'][i * 7, :, :, 0], cmap='gray', vmin=-2, vmax=2)
        ax.axis('off')
        ax.set_title(f"f{i*7}", color='white', fontsize=8)
    plt.suptitle(f"'{sample['label']}'", color='white', fontsize=10, y=1.06)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/sanity_check.png', dpi=130, bbox_inches='tight', facecolor='#0d0d1a')
    plt.show()

    # pixel distribution
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(sample['frames'].flatten(), bins=60, color='#2196F3', alpha=0.85, edgecolor='none')
    ax.axvline(0, color='#FFD700', linewidth=1.5, linestyle='--', label='mean = 0')
    ax.set_title("pixel distribution (normalized)", color='white')
    ax.set_xlabel("pixel value", color='white')
    ax.tick_params(colors='white')
    ax.legend(labelcolor='white', facecolor='#16213e', edgecolor='#2a2a4a')
    ax.set_facecolor('#16213e')
    fig.patch.set_facecolor('#0d0d1a')
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/pixel_dist.png', dpi=130, bbox_inches='tight', facecolor='#0d0d1a')
    plt.show()
    print("sanity check done.")

In [ ]:
def main():

    # 1 for the setup
    mount_drive()
    install_deps()
    download_dlib_predictor()
    setup_kaggle()
    download_grid()
    start_heartbeat()

    # 2 for the speakers
    available = list_available_speakers()
    global speakers  # training needs this for checkpoint metadata
    speakers = select_speakers(available)
    train_speakers, val_speaker = split_train_val(speakers)

    # 3 for the pre-processing
    detector, predictor = load_dlib_models()
    preprocess_all(speakers, detector, predictor)

    # 4 for the verification
    count_files_per_speaker()
    sanity_check()

    # 5 for the dataloaders
    train_loader, val_loader = build_dataloaders(speakers, val_speaker)